# Interactive mass model

Self-contained playground for the planetesimal mass model: every function below is a copy of the pipeline's code
(`src/mass_models/` and `src/utilities/semimajor_axis.py`), so you can edit them here and see what changes. Nothing is imported from the repo.

1. Draw a big size **cascade** `dN/dR ∝ R^-q` on `[R_min, R_max]` (`sample_cascade_top`).
2. **Select** `n_keep` bodies: drop the `n_drop` largest (`drop_top`), or keep the block that best traces the analytic power law (`powerlaw`).
3. Convert radii → masses with a size-dependent (or constant) density.
4. Optionally **rescale** every mass by one constant so the disk totals a chosen mass (shown as a second mass-vs-a plot).
5. Draw each body's **semimajor axis** (uniform or Lorentzian in ln a) exactly as `run_simulation` does, and plot mass vs a.

Run the cells top to bottom, then change the sliders in section 7 and press **Run**.

In [ ]:
import io
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.transforms as mtransforms
import ipywidgets as w
from IPython.display import display, clear_output

%matplotlib inline

## 1. Constants, density and unit conversions  *(from `mass_models.py`)*

In [ ]:
M_SUN_KG = 1.98892e30
M_EARTH_KG = M_SUN_KG * 3.0034896149156e-6

DENSITY_R0_KM = 220.0
DENSITY_MAX_G_CM3 = 2.1


def size_dependent_density(radii_km, r0_km=DENSITY_R0_KM, rho_max_g_cm3=DENSITY_MAX_G_CM3):
    """rho(R) = [ (R / r0)^-3 + rho_max^-3 ]^(-1/3), g/cm^3.
    Porous small bodies (rho ~ R / r0) grading into a compact rho_max plateau."""
    radii_km = np.asarray(radii_km, dtype=float)
    return ((radii_km / r0_km) ** -3 + rho_max_g_cm3 ** -3) ** (-1.0 / 3.0)


def radii_to_masses(radii_km, density_g_cm3=1.0):
    """m = (4/3) pi rho R^3. Radii in km, density in g/cm^3 (scalar or per body); returns kg."""
    radii_m = np.asarray(radii_km, dtype=float) * 1000.0
    return (4.0 / 3.0) * np.pi * (density_g_cm3 * 1000.0) * radii_m**3


def sample_powerlaw(n_particles, value_min, value_max, slope, seed=None):
    """Sample dN/dx ∝ x^-slope on [value_min, value_max] by the analytic inverse CDF."""
    if n_particles <= 0:
        raise ValueError("n_particles must be > 0")
    if value_min <= 0:
        raise ValueError("value_min must be > 0")
    if value_max <= value_min:
        raise ValueError("value_max must be greater than value_min")
    rng = seed if isinstance(seed, np.random.Generator) else np.random.default_rng(seed)
    u = rng.uniform(0.0, 1.0, n_particles)
    if np.isclose(slope, 1.0):
        return value_min * (value_max / value_min) ** u
    e = 1.0 - slope
    return (u * (value_max**e - value_min**e) + value_min**e) ** (1.0 / e)

## 2. Cascade sampling and selection  *(from `make_cascade_selection.py`)*

In [ ]:
DENSITY_MODELS = ("size_dependent", "constant")
SELECTION_MODES = ("drop_top", "powerlaw")
_SAMPLE_CHUNK = 5_000_000
_CLEAN_COUNT = 1000.0  # beyond this expected count Poisson scatter << tol


@dataclass
class Cfg:
    n_cascade: int
    n_keep: int
    radius_min: float
    radius_max: float
    density_g_cm3: float
    tol: float
    seed: int
    selection: str = "drop_top"
    n_drop: int = 100
    total_mass_earth: float = None
    density_model: str = "size_dependent"

    def densities(self, radii_km):
        """Bulk density (g/cm^3) of each radius under ``density_model``."""
        if self.density_model == "size_dependent":
            return size_dependent_density(radii_km)
        return np.full(np.shape(radii_km), float(self.density_g_cm3))

    @property
    def keep_top(self):
        """How many of the largest cascade radii to retain in memory."""
        return max(20_000, 25 * self.n_keep)


def analytic_cumulative(cfg, radii_km, slope, n):
    """Analytic N(>=R) for dN/dR ~ R^-q on [radius_min, radius_max], scaled so N(>=radius_min) = n."""
    r = np.asarray(radii_km, dtype=float)
    rmin, rmax = cfg.radius_min, cfg.radius_max
    if np.isclose(slope, 1.0):
        num, den = np.log(rmax / r), np.log(rmax / rmin)
    else:
        p = 1.0 - slope
        num, den = r ** p - rmax ** p, rmin ** p - rmax ** p
    return n * num / den


def sample_cascade_top(cfg, slope, seed):
    """Draw ``cfg.n_cascade`` radii from dN/dR ~ R^-slope in chunks, keeping only the
    ``cfg.keep_top`` largest. Returns them sorted largest -> smallest."""
    rng = np.random.default_rng(seed)
    kept = np.empty(0, dtype=float)
    remaining = int(cfg.n_cascade)
    keep_top = cfg.keep_top
    while remaining > 0:
        m = min(_SAMPLE_CHUNK, remaining)
        remaining -= m
        r = sample_powerlaw(m, cfg.radius_min, cfg.radius_max, float(slope), seed=rng)
        if kept.size:
            r = np.concatenate([kept, r])
        if r.size > keep_top:
            r = np.partition(r, r.size - keep_top)[-keep_top:]
        kept = r
    return np.sort(kept)[::-1]


def find_cut_rank(cfg, radii_desc, slope):
    """First rank (1-based) after which the next ``n_keep`` bodies sit within ``cfg.tol``
    of the analytic power law, i.e. drop the whole deviant tail. Returns (k_start, ratio).

    What ``tol`` does (selection="powerlaw" only; "drop_top" ignores it and just drops ``n_drop``):
    the largest bodies of a finite draw scatter away from the analytic N(>=R) because there are
    few of them. ``ratio`` = sampled N(>=R) / analytic N(>=R); a rank is *deviant* if
    |ratio - 1| > tol. Only the small-count regime is checked (expected count < _CLEAN_COUNT).
    The kept block starts one rank past the deepest deviant rank.
      smaller tol -> stricter -> deeper cut, more big bodies dropped, smaller bodies kept
      larger tol  -> looser   -> cut moves toward rank 1, noisier big bodies can be kept
    The gray band (1 +/- tol) and red cut line in the section-6 departure plot show this."""
    counts = np.arange(1, radii_desc.size + 1, dtype=float)
    expected = analytic_cumulative(cfg, radii_desc, slope, cfg.n_cascade)
    ratio = counts / expected

    max_start = radii_desc.size - cfg.n_keep + 1
    if max_start < 1:
        raise ValueError(f"retained {radii_desc.size} ranks but need n_keep={cfg.n_keep}; "
                         "raise keep_top / n_cascade")
    horizon = min(int(np.searchsorted(expected, _CLEAN_COUNT)), max_start)
    bad_idx = np.where(np.abs(ratio[:horizon] - 1.0) > cfg.tol)[0]
    k_start = int(bad_idx[-1]) + 2 if bad_idx.size else 1
    return min(k_start, max_start), ratio


def select_start_rank(cfg, radii_desc, slope):
    """Start rank (1-based) of the kept block for ``cfg.selection``, plus the
    sampled / analytic ratio used by the diagnostic plot."""
    if cfg.selection == "powerlaw":
        return find_cut_rank(cfg, radii_desc, slope)
    if radii_desc.size < cfg.n_drop + cfg.n_keep:
        raise ValueError(f"retained {radii_desc.size} ranks but need n_drop + n_keep = "
                         f"{cfg.n_drop + cfg.n_keep}; raise keep_top / n_cascade")
    counts = np.arange(1, radii_desc.size + 1, dtype=float)
    ratio = counts / analytic_cumulative(cfg, radii_desc, slope, cfg.n_cascade)
    return cfg.n_drop + 1, ratio


def build_kept_frame(cfg, radii_desc, slope, k_start, seed):
    """DataFrame for the kept block (same columns as selected.csv)."""
    block = radii_desc[k_start - 1 : k_start - 1 + cfg.n_keep].copy()
    density = cfg.densities(block)
    mass_kg = radii_to_masses(block, density_g_cm3=density)
    # Optional rescale to a target disk mass: one constant factor on every mass, so the
    # spectrum keeps its shape (radius/density stay as drawn, so m != 4/3 pi rho R^3 then).
    mass_scale = 1.0
    if cfg.total_mass_earth is not None:
        mass_scale = cfg.total_mass_earth * M_EARTH_KG / mass_kg.sum()
        mass_kg = mass_kg * mass_scale
    df = pd.DataFrame({
        "particle_id": np.arange(cfg.n_keep),
        "radius_km": block,
        "density_g_cm3": density,
        "mass_kg": mass_kg,
        "mass_earth": mass_kg / M_EARTH_KG,
        "mass_solar": mass_kg / M_SUN_KG,
    })
    df.attrs.update(slope=float(slope), seed=seed, mass_scale=mass_scale,
                    rank_window=(k_start, k_start + cfg.n_keep - 1))
    return df

def rescale_masses(kept, total_mass_earth):
    """Copy of ``kept`` with every mass multiplied by one constant so they sum to
    ``total_mass_earth`` (Earth masses). Same math as ``Cfg.total_mass_earth`` in
    ``build_kept_frame``, so the spectrum keeps its shape; radius/density are unchanged."""
    out = kept.copy()
    scale = total_mass_earth * M_EARTH_KG / out["mass_kg"].sum()
    out["mass_kg"] *= scale
    out["mass_earth"] = out["mass_kg"] / M_EARTH_KG
    out["mass_solar"] = out["mass_kg"] / M_SUN_KG
    out.attrs["mass_scale"] = scale * kept.attrs.get("mass_scale", 1.0)
    return out

## 3. Semimajor axes  *(from `semimajor_axis.py` and `plot_mass_vs_semimajor_axis.py`)*

Each body gets one `a`. The draw order per body is a, e, inc, ω, Ω, M, so the six uniform calls below must stay in step with `run_simulation.build_simulation` (seed 42) to reproduce the locations the real runs use.

In [ ]:
A_DISTRIBUTIONS = ("uniform", "lorentzian")
SIM_SEED = 42  # hardcoded in run_simulation.build_simulation


def lorentzian_log_params(mu_0, sigma):
    """(ln mu, gamma) of the log-space Lorentzian for disk center/std mu_0, sigma (AU)."""
    mu_0, sigma = float(mu_0), float(sigma)
    if mu_0 <= 0 or sigma <= 0:
        raise ValueError(f"a_center and a_sigma must be > 0, got {mu_0}, {sigma}")
    ln_mu = np.log(mu_0**2 / np.sqrt(mu_0**2 + sigma**2))
    gamma = np.sqrt(np.log1p(sigma**2 / mu_0**2))
    return ln_mu, gamma


class SemimajorAxisSampler:
    """Draws initial semimajor axes (AU) from a ``disk`` dict."""

    def __init__(self, disk):
        self.amin = float(disk["amin"])
        self.amax = float(disk["amax"])
        self.kind = disk.get("a_distribution", "uniform")
        if self.kind not in A_DISTRIBUTIONS:
            raise ValueError(f"a_distribution must be one of {A_DISTRIBUTIONS}, got {self.kind!r}")
        if self.kind == "lorentzian":
            self.mu_0 = float(disk["a_center"])
            self.sigma = float(disk["a_sigma"])
            self.ln_mu, self.gamma = lorentzian_log_params(self.mu_0, self.sigma)
            # inverse-CDF angles of the truncation bounds in ln a
            self._theta_lo = np.arctan((np.log(self.amin) - self.ln_mu) / self.gamma)
            self._theta_hi = np.arctan((np.log(self.amax) - self.ln_mu) / self.gamma)

    @property
    def peak(self):
        return float(np.exp(self.ln_mu)) if self.kind == "lorentzian" else None

    def draw(self, rng):
        if self.kind == "uniform":
            return rng.uniform(self.amin, self.amax)
        theta = rng.uniform(self._theta_lo, self._theta_hi)
        return float(np.exp(self.ln_mu + self.gamma * np.tan(theta)))

    def describe(self):
        if self.kind == "uniform":
            return f"uniform in [{self.amin:g}, {self.amax:g}] AU"
        return (f"Lorentzian in ln a, mu_0={self.mu_0:g}, sigma={self.sigma:g} AU "
                f"-> peak {self.peak:.6g} AU, truncated to [{self.amin:g}, {self.amax:g}] AU")


def semimajor_axes(n, disk, seed=SIM_SEED):
    """Initial a (AU) of each planetesimal, as run_simulation draws them."""
    sampler = SemimajorAxisSampler(disk)
    emax = float(disk["emax"])
    imax = np.radians(float(disk["imax_deg"]))
    rng = np.random.default_rng(seed)
    a = np.empty(n)
    for i in range(n):
        a[i] = sampler.draw(rng)
        rng.uniform(0.0, emax)
        rng.uniform(0.0, imax)
        rng.uniform(0, 2 * np.pi, size=3)
    return a

## 4. Solar System reference lines  *(from `reference_bodies.py`)*

In [ ]:
# name: (mass [M_earth], mean radius [km])
REFERENCE_BODIES = {
    "Jupiter": (317.83, 69911.0), "Neptune": (17.147, 24622.0), "Earth": (1.0, 6371.0),
    "Mars": (0.10745, 3389.5), "Mercury": (0.05527, 2439.7), "Moon": (0.012300, 1737.4),
    "Pluto": (0.002187, 1188.3), "Ceres": (1.574e-4, 469.7), "Vesta": (4.34e-5, 262.7),
    "Phoebe": (1.388e-6, 106.5), "Ida": (7.03e-9, 15.7), "Phobos": (1.785e-9, 11.1),
    "Arrokoth": (1.26e-10, 9.0),
}
DEFAULT_PAD_FACTOR = 2.0
_LINE_KW = dict(color="0.35", lw=0.8, ls="--", alpha=0.6, zorder=2.5)
_TEXT_KW = dict(color="0.25", zorder=6,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))


def _add_references(ax, values, axis, pad_factor, label_pos, fontsize):
    lo, hi = ax.get_ylim() if axis == "y" else ax.get_xlim()
    lo, hi = min(lo, hi), max(lo, hi)
    shown = {n: v for n, v in values.items() if lo / pad_factor <= v <= hi * pad_factor}
    if not shown:
        return
    new_lo = min(lo, min(shown.values()) / 1.15)
    new_hi = max(hi, max(shown.values()) * 1.15)
    text_kw = dict(_TEXT_KW, fontsize=fontsize)
    for name, v in shown.items():
        if axis == "y":
            ax.axhline(v, **_LINE_KW)
            trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
            ax.text(label_pos, v, name, transform=trans, ha="right", va="center", **text_kw)
        else:
            ax.axvline(v, **_LINE_KW)
            trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
            ax.text(v, label_pos, name, transform=trans, ha="center", va="top", rotation=90, **text_kw)
    (ax.set_ylim if axis == "y" else ax.set_xlim)(new_lo, new_hi)


def add_mass_references(ax, axis="y", pad_factor=DEFAULT_PAD_FACTOR, label_pos=0.99, fontsize=7):
    """Mark reference-body masses (Earth masses) on a log mass axis."""
    _add_references(ax, {n: m for n, (m, r) in REFERENCE_BODIES.items()}, axis, pad_factor, label_pos, fontsize)


def add_radius_references(ax, axis="x", pad_factor=DEFAULT_PAD_FACTOR, label_pos=0.99, fontsize=7):
    """Mark reference-body radii (km) on a log radius axis."""
    _add_references(ax, {n: r for n, (m, r) in REFERENCE_BODIES.items()}, axis, pad_factor, label_pos, fontsize)

## 5. Build a mass model

`MODEL_DEFAULTS` is the single source of truth for the model settings: the sliders in section 7 start from these values and
`build_mass_model` uses them as its defaults, so the two can't drift apart. After you press **Run** in section 7, the globals
`kept`, `kept_rescaled` and `cfg` hold the model built from the *slider* values.

In [ ]:
# Note on ``tol``: only used when selection="powerlaw" (see find_cut_rank). It is the largest fractional deviation of the
# sampled N(>=R) from the analytic power law that is tolerated; ranks deviating more are dropped from the large end.
# Smaller tol = stricter = deeper cut (smaller bodies kept). With selection="drop_top" it is ignored.
MODEL_DEFAULTS = dict(
    q=3.5, radius_min=1.0, radius_max=1000.0, n_keep=800, selection="drop_top", n_drop=100,
    tol=0.2, n_cascade=5_000_000, density_model="size_dependent", density=1.0, seed=42,
)


def build_mass_model(**overrides):
    """Build one mass model. Any setting not given falls back to MODEL_DEFAULTS
    (plus ``total_mass_earth``: rescale to this disk mass, default None).
    Returns (kept DataFrame, cfg, radii_desc, k_start, ratio)."""
    bad = set(overrides) - set(MODEL_DEFAULTS) - {"total_mass_earth"}
    if bad:
        raise TypeError(f"unknown mass-model settings: {sorted(bad)}")
    p = {**MODEL_DEFAULTS, "total_mass_earth": None, **overrides}
    cfg = Cfg(
        n_cascade=int(p["n_cascade"]), n_keep=int(p["n_keep"]),
        radius_min=p["radius_min"], radius_max=p["radius_max"],
        density_g_cm3=p["density"], tol=p["tol"], seed=int(p["seed"]), selection=p["selection"],
        n_drop=int(p["n_drop"]), total_mass_earth=p["total_mass_earth"], density_model=p["density_model"],
    )
    radii_desc = sample_cascade_top(cfg, float(p["q"]), int(p["seed"]))
    k_start, ratio = select_start_rank(cfg, radii_desc, float(p["q"]))
    kept = build_kept_frame(cfg, radii_desc, float(p["q"]), k_start, int(p["seed"]))
    return kept, cfg, radii_desc, k_start, ratio

## 6. Plotting

In [ ]:
def plot_model(kept, cfg, radii_desc, k_start, ratio, q, title=None):
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    ax = ax.ravel()
    lo, hi = k_start, k_start + cfg.n_keep - 1
    counts = np.arange(1, radii_desc.size + 1)

    # (a) cumulative N(>=R): dropped / kept / below vs analytic
    a = ax[0]
    if lo > 1:
        a.plot(radii_desc[:lo-1], counts[:lo-1], "o", ms=3, alpha=.55, c="tab:red", label="dropped (largest)")
    a.plot(radii_desc[lo-1:hi], counts[lo-1:hi], "o", ms=3, alpha=.6, c="tab:green", label="kept")
    a.plot(radii_desc[hi:], counts[hi:], "o", ms=2, alpha=.3, c="0.6", label="below block")
    r_line = np.logspace(np.log10(0.7*radii_desc[hi-1]), np.log10(cfg.radius_max), 300)
    a.plot(r_line, analytic_cumulative(cfg, r_line, q, cfg.n_cascade), c="0.2", lw=1.3, label="analytic")
    a.set(xscale="log", yscale="log", xlabel="Radius (km)", ylabel="N(≥R)", title="Cumulative size distribution")
    a.set_xlim(0.8*radii_desc[hi-1], cfg.radius_max*1.03); a.set_ylim(0.7, 6*hi)
    add_radius_references(a, axis="x", pad_factor=1.0, fontsize=6)
    a.xaxis.set_major_formatter(mticker.ScalarFormatter()); a.xaxis.set_minor_formatter(mticker.NullFormatter())
    a.legend(fontsize=8, loc="lower left"); a.grid(alpha=.3, which="both")

    # (b) departure from the power law
    b = ax[1]
    ranks = np.arange(1, ratio.size + 1)
    if cfg.selection == "powerlaw":
        b.axhspan(1-cfg.tol, 1+cfg.tol, color="0.85", zorder=0)
    b.axhline(1, c="0.4", lw=.8)
    b.plot(ranks, ratio, "o", ms=2, alpha=.5)
    b.axvspan(lo, hi, color="tab:green", alpha=.18); b.axvline(lo, c="tab:red")
    b.set(xscale="log", ylim=(0, 2), xlabel="Rank", ylabel="sampled / analytic N(≥R)",
          title=f"Finite-sample departure (kept ranks {lo}–{hi})")
    b.grid(alpha=.3, which="both")

    # (c) rank vs mass
    ordered = kept.sort_values("mass_kg", ascending=False).reset_index(drop=True)
    rank = np.arange(1, len(ordered)+1)
    c = ax[2]
    c.scatter(ordered["mass_earth"], rank, s=12)
    c.set(xscale="log", yscale="log", xlabel="Mass (Earth masses)", ylabel="Rank (most massive = 1)",
          title="Individual masses")
    add_mass_references(c, axis="x"); c.grid(alpha=.3, which="both")

    # (d) mass per log-radius bin (where the mass lives) + density
    d = ax[3]
    bins = np.logspace(np.log10(kept.radius_km.min()), np.log10(kept.radius_km.max()), 25)
    mass_in_bin, _ = np.histogram(kept.radius_km, bins=bins, weights=kept.mass_earth)
    d.stairs(mass_in_bin, bins, fill=True, alpha=.5)
    d.set(xscale="log", yscale="log", xlabel="Radius (km)", ylabel="Mass in bin (Earth masses)",
          title="Where the mass is")
    d.grid(alpha=.3, which="both")
    d2 = d.twinx()
    r = np.logspace(np.log10(kept.radius_km.min()), np.log10(kept.radius_km.max()), 100)
    d2.plot(r, cfg.densities(r), c="tab:orange"); d2.set_ylabel("ρ (g/cm³)", color="tab:orange")
    if title:
        fig.suptitle(title, fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
    else:
        fig.tight_layout()
    return fig


def plot_mass_vs_a(kept, a, q, disk_desc, uniform_control=True, tag=""):
    """Mass vs initial semimajor axis. Bodies are paired with a-locations by particle_id,
    as in plot_mass_vs_semimajor_axis.py (the a draws don't depend on mass)."""
    fig, ax = plt.subplots(figsize=(8.5, 6))
    ax.scatter(a, kept["mass_earth"], s=10, alpha=0.65, label=f"q = {q:g}")
    if uniform_control:
        # control: every body gets the same mass, M_total / N
        m_uniform = float(kept["mass_earth"].sum()) / len(kept)
        ax.scatter(a, np.full(len(a), m_uniform), s=6, marker="s", color="tab:red", alpha=0.7,
                   label=f"uniform control: M_total/N = {m_uniform:.3g} M_earth ({len(a)} bodies)")
    ax.set_yscale("log")
    add_mass_references(ax)
    ax.set_xlabel("Semimajor axis (AU)"); ax.set_ylabel("Mass (Earth masses)")
    ax.set_title(f"Planetesimal mass vs. semimajor axis{tag}\n{disk_desc}", fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
    return fig


def summarize(kept, cfg, q):
    """Text summary of one mass model."""
    a = kept.attrs
    return "\n".join([
        f"q = {q:g}   selection = {cfg.selection}   cascade N = {cfg.n_cascade:,}   kept = {len(kept)}",
        f"radius : {kept.radius_km.min():.4g} – {kept.radius_km.max():.4g} km",
        f"mass   : {kept.mass_earth.min():.3e} – {kept.mass_earth.max():.3e} M_earth",
        f"total  : {kept.mass_earth.sum():.4g} M_earth   (mass_scale = {a['mass_scale']:.4g})",
        f"largest body holds {kept.mass_earth.max()/kept.mass_earth.sum():.1%} of the mass",
    ])


def fig_png(fig):
    """Render a figure to PNG bytes (and close it) for an ipywidgets Image."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()

## 7. Interactive controls

Sliders only update the settings; everything is redrawn when you press **Run** (a 5M-body cascade takes a few seconds).
Slider starting values come from `MODEL_DEFAULTS` (section 5); the disk defaults match
`config/SS_800MP_500kyr_1Mearth_slopeq3.5_lorentzian.yaml`. `slider_params()` returns the current slider settings, ready for `build_mass_model(**slider_params())`.

In [ ]:
style = {"description_width": "130px"}
lay = w.Layout(width="440px")
def fs(v, lo, hi, step, desc): return w.FloatSlider(value=v, min=lo, max=hi, step=step, description=desc, style=style, layout=lay, continuous_update=False)
def isl(v, lo, hi, step, desc): return w.IntSlider(value=v, min=lo, max=hi, step=step, description=desc, style=style, layout=lay, continuous_update=False)

M = MODEL_DEFAULTS
# mass-model controls
W = dict(
    q=fs(M["q"], 1.5, 5.5, 0.1, "slope q"),
    radius_min=fs(M["radius_min"], 0.1, 50, 0.1, "R min (km)"),
    radius_max=fs(M["radius_max"], 50, 3000, 10, "R max (km)"),
    n_keep=isl(M["n_keep"], 50, 5000, 50, "n_keep"),
    selection=w.ToggleButtons(options=list(SELECTION_MODES), value=M["selection"], description="selection", style=style),
    n_drop=isl(M["n_drop"], 0, 1000, 10, "n_drop"),
    # tol: powerlaw selection only -- max fractional deviation from the analytic power law tolerated in the kept block.
    # Smaller = stricter = deeper cut (more large bodies dropped). Ignored by drop_top. See find_cut_rank.
    tol=fs(M["tol"], 0.05, 0.5, 0.05, "tol (powerlaw)"),
    n_cascade=w.Dropdown(options=[100_000, 500_000, 1_000_000, 5_000_000, 10_000_000], value=M["n_cascade"], description="n_cascade", style=style),
    density_model=w.ToggleButtons(options=list(DENSITY_MODELS), value=M["density_model"], description="density", style=style),
    density=fs(M["density"], 0.5, 3.5, 0.1, "ρ const (g/cm³)"),
    total_mass_earth=fs(1.0, 0.01, 20, 0.01, "M_disk (rescaled plot)"),
    seed=isl(M["seed"], 0, 999, 1, "seed"),
)
# disk / semimajor-axis controls
D = dict(
    a_distribution=w.ToggleButtons(options=list(A_DISTRIBUTIONS), value="lorentzian", description="a distribution", style=style),
    amin=fs(95, 1, 500, 1, "a min (AU)"),
    amax=fs(105, 2, 1000, 1, "a max (AU)"),
    a_center=fs(100, 1, 500, 1, "a center (AU)"),
    a_sigma=fs(1.0, 0.1, 50, 0.1, "a sigma (AU)"),
    emax=w.FloatText(value=3.2e-5, description="e max", style=style),
    imax_deg=w.FloatText(value=3.2e-5, description="i max (deg)", style=style),
    a_seed=isl(SIM_SEED, 0, 999, 1, "a seed"),
)
run_btn = w.Button(description="Run", button_style="primary", icon="play")
# Results live in plain widgets (no Output widget / clear_output), so the controls above are never wiped.
status = w.HTML()
img_model = w.Image(format="png", layout=w.Layout(max_width="100%"))
img_a = w.Image(format="png", layout=w.Layout(max_width="100%"))
img_a_rescaled = w.Image(format="png", layout=w.Layout(max_width="100%"))
state = {}


def slider_params():
    """Current mass-model slider settings (everything build_mass_model takes except total_mass_earth)."""
    return {k: v.value for k, v in W.items() if k != "total_mass_earth"}


def m_disk():
    """Current M_disk slider value (Earth masses), used for every 'rescaled' plot."""
    return W["total_mass_earth"].value


def current_disk():
    return {k: v.value for k, v in D.items() if k != "a_seed"}


def run(_=None):
    global kept, kept_rescaled, cfg
    p, md = slider_params(), m_disk()
    status.value = "<i>running…</i>"
    try:
        kept, cfg, rd, ks, ratio = build_mass_model(**p)
        disk = current_disk()
        a = semimajor_axes(len(kept), disk, seed=D["a_seed"].value)
        desc = SemimajorAxisSampler(disk).describe()
        kept_rescaled = rescale_masses(kept, md)
        state.update(kept=kept, kept_rescaled=kept_rescaled, cfg=cfg, params=p, m_disk=md, a=a, disk=disk)
        img_model.value = fig_png(plot_model(kept, cfg, rd, ks, ratio, p["q"]))
        img_a.value = fig_png(plot_mass_vs_a(kept, a, p["q"], desc))
        img_a_rescaled.value = fig_png(plot_mass_vs_a(
            kept_rescaled, a, p["q"], desc, tag=f" (masses rescaled to M_disk = {md:g} M_earth)"))
        status.value = f"<pre>{summarize(kept, cfg, p['q'])}\n{desc}</pre>"
    except Exception as e:
        status.value = f"<pre style='color:crimson'>Error: {e}</pre>"

run_btn.on_click(run)
display(w.VBox([w.HTML("<b>Mass model</b>"), *W.values(), w.HTML("<b>Disk / semimajor axis</b>"), *D.values(),
                run_btn, status, img_model, img_a, img_a_rescaled]))
run()

## 8. Save what you made

Writes the current model to a CSV: the same columns as `selected.csv`, plus the semimajor axis `a_au` paired with each body
and `mass_earth_rescaled` (masses rescaled to the M_disk slider).

In [ ]:
name = "my_mass_model"          # <- change me
out_path = f"{name}.csv"        # saved next to the notebook
df = state["kept"].copy()
df["a_au"] = state["a"]
df["mass_earth_rescaled"] = state["kept_rescaled"]["mass_earth"]
df.to_csv(out_path, index=False)
print("Saved:", out_path)
print("mass-model params:", state["params"], " M_disk:", state["m_disk"])
print("disk:", state["disk"])

## 9. Compare several slopes

The selections are built **once** (`build_slope_sets`: slider settings for everything except `q`; slope *i* uses seed + *i*, as the script does) and then plotted, so the
raw and rescaled views show the *same* bodies. They are kept in the global `slope_sets` and **saved to disk** by `save_slope_sets`
in the pipeline's layout, `mass_model_exports/<name>/slope_q<q>/selected.csv`, plus `selection_summary.csv` and `params.json`
(the settings used), so you can reload, diff or re-plot a comparison later. Set `SAVE_AS = None` to skip saving.

In [ ]:
import json
from pathlib import Path


def build_slope_sets(slopes, params=None, base_seed=None):
    """{q: kept DataFrame, masses as drawn}. ``params`` default to the sliders; slope i uses seed base_seed + i
    (base_seed defaults to params['seed'])."""
    p = {**MODEL_DEFAULTS, **(slider_params() if params is None else params)}
    p.pop("q", None)
    base = p.pop("seed") if base_seed is None else base_seed
    p.pop("total_mass_earth", None)
    sets = {}
    for i, q in enumerate(slopes):
        kept_q, *_ = build_mass_model(q=q, seed=base + i, **p)
        sets[float(q)] = kept_q
    return sets


def save_slope_sets(sets, name, a, md, params, disk, a_seed, outdir="mass_model_exports"):
    """Write each slope's selection like make_cascade_selection.py (slope_q<q>/selected.csv + selection_summary.csv)
    with the semimajor axis ``a_au`` and ``mass_earth_rescaled`` (rescaled to ``md``) added, plus params.json."""
    root = Path(outdir) / name
    rows = []
    for q, k in sets.items():
        d = root / f"slope_q{q:g}"
        d.mkdir(parents=True, exist_ok=True)
        df = k.copy()
        df["a_au"] = a
        df["mass_earth_rescaled"] = rescale_masses(k, md)["mass_earth"]
        df.to_csv(d / "selected.csv", index=False)
        rows.append(dict(slope=q, cut_rank=k.attrs["rank_window"][0], keep_r_min=k.radius_km.min(),
                         keep_r_max=k.radius_km.max(), total_mass_earth=k.mass_earth.sum(),
                         mass_scale=k.attrs["mass_scale"], m_disk_rescaled=md))
    pd.DataFrame(rows).to_csv(root / "selection_summary.csv", index=False)
    with open(root / "params.json", "w") as f:
        json.dump(dict(model=params, slopes=list(sets), m_disk_rescaled=md, disk=disk, a_seed=a_seed),
                  f, indent=2, default=str)
    print("Saved selections to:", root)
    return root


def compare_slopes(slopes=(2.5, 3.0, 3.5, 4.0, 4.5), rescaled=False, sets=None, save_as=None):
    """Overlay several slopes built from the slider settings. rescaled=True rescales every slope to the M_disk slider.
    Pass ``sets`` to re-plot selections you already built; ``save_as`` writes them to mass_model_exports/<save_as>/."""
    global slope_sets
    params, md, disk = slider_params(), m_disk(), current_disk()
    if sets is None:
        sets = build_slope_sets(slopes, params)
    slope_sets = sets
    n_bodies = len(next(iter(sets.values())))
    a = semimajor_axes(n_bodies, disk, seed=D["a_seed"].value)
    state.update(sets=sets, sets_a=a)
    shown = {q: rescale_masses(k, md) for q, k in sets.items()} if rescaled else sets

    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    cm = plt.get_cmap("viridis")
    rows = []
    for i, (q, k) in enumerate(shown.items()):
        col = cm(i / max(1, len(shown) - 1))
        o = k.sort_values("radius_km", ascending=False)
        n = np.arange(1, len(o) + 1)
        ax[0].plot(o.radius_km, n, c=col, label=f"q={q:g}")
        ax[1].plot(o.mass_earth, n, c=col, label=f"q={q:g}")
        ax[2].scatter(a, k.mass_earth, s=8, alpha=.65, color=col, label=f"q={q:g}")
        ax[2].axhline(k.mass_earth.sum() / len(k), ls="--", lw=1.2, color=col)
        rows.append(dict(q=q, r_min=k.radius_km.min(), r_max=k.radius_km.max(),
                         total_mass_earth=k.mass_earth.sum(), top_body_frac=k.mass_earth.max()/k.mass_earth.sum()))
    for x, xl in zip(ax[:2], ["Radius (km)", "Mass (Earth masses)"]):
        x.set(xscale="log", yscale="log", xlabel=xl, ylabel="N(≥x)"); x.grid(alpha=.3, which="both"); x.legend()
    ax[2].plot([], [], ls="--", c="0.3", label="M_total/N (dashed, per slope)")
    ax[2].set(yscale="log", xlabel="Semimajor axis (AU)", ylabel="Mass (Earth masses)", title="Mass vs a" + (" (rescaled to M_disk)" if rescaled else ""))
    add_mass_references(ax[2]); ax[2].grid(alpha=.3); ax[2].legend(fontsize=7)
    plt.tight_layout(); plt.show()
    if save_as:
        save_slope_sets(sets, save_as, a, md, params, disk, D["a_seed"].value)
    return pd.DataFrame(rows)

In [ ]:
SAVE_AS = "compare_slopes"   # folder under mass_model_exports/; None = don't save
compare_slopes(save_as=SAVE_AS)
# Same selections, rescaled to the M_disk slider (no redraw):
# compare_slopes(rescaled=True, sets=slope_sets)

## 10. Multi-slope mass vs a with the uniform M_disk/N control

This is the logic of `plot_mass_vs_semimajor_axis.py` (several slopes on one mass-vs-a plot), with the uniform control set to **M_disk / N**:

1. **One cascade selection per slope** (`build_slope_sets`): slope `q_i` uses seed `42 + i` (as `make_cascade_selection.py` does).
   With `USE_SLIDERS = False` this is the fixed `MODEL_DEFAULTS` configuration (800 bodies ranked 101–900 of a 5M-body cascade on R ∈ [1, 1000] km, size-dependent density).
2. **Rescale each slope to the same disk mass** (1 M⊕ by default): one constant per slope, so each spectrum keeps its shape but all slopes total the same.
3. **One shared set of semimajor axes.** The a-draws don't depend on mass, so every slope uses the same locations.
4. **Overlay the slopes** as mass vs a, body *i* of every slope paired with location *i*.
5. **Uniform control.** Every location gets the same mass, M_disk / N (red squares). Because every slope is rescaled to the same M_disk, one line serves all slopes.

With `USE_SLIDERS = True`, step 1 uses the section-7 slider values (all but `q`) and M_disk comes from the slider. The selections are saved like in section 9 (set `SAVE_AS = None` to skip).

`plot_mass_vs_a_multi` is the plot.

In [ ]:
def plot_mass_vs_a_multi(sets, a, total_mass_earth=None, title_extra=""):
    """Mass vs semimajor axis for several slopes + the uniform control, M_total / N (red).
    If the slopes don't all total the same mass, each gets its own dashed M_total/N line instead."""
    n = len(a)
    assert all(len(df) == n for df in sets.values()), "every slope needs the same number of bodies as locations"
    totals = {q: float(df["mass_earth"].sum()) for q, df in sets.items()}
    same_total = np.allclose(list(totals.values()), next(iter(totals.values())), rtol=1e-6)

    fig, ax = plt.subplots(figsize=(8.5, 6))
    for q, df in sets.items():
        sc = ax.scatter(a, df["mass_earth"], s=10, alpha=0.65, label=f"q = {q:g}")
        if not same_total:
            ax.axhline(totals[q] / n, ls="--", lw=1.2, color=sc.get_facecolor()[0])
    if same_total:
        m_uniform = next(iter(totals.values())) / n
        ax.scatter(a, np.full(n, m_uniform), s=6, marker="s", color="tab:red", alpha=0.7,
                   label=f"uniform: M_disk/N = {m_uniform:.3g} M_earth ({n} bodies)")
        ctrl = f"uniform M_disk/N ({m_uniform:.4g} M_earth)"
    else:
        ax.plot([], [], ls="--", c="0.3", label="M_total/N (dashed, per slope)")
        ctrl = "per-slope M_total/N (dashed)"
    ax.set_yscale("log")
    add_mass_references(ax)
    ax.set_xlabel("Semimajor axis (AU)"); ax.set_ylabel("Mass (Earth masses)")
    resc = f" (slopes rescaled to M_disk = {total_mass_earth:g} M_earth)" if total_mass_earth else ""
    ax.set_title(f"Planetesimal mass vs. semimajor axis{resc}\n"
                 f"{len(sets)} slope distributions + {ctrl}, same {n} locations{title_extra}", fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
    for q, t in totals.items():
        print(f"q = {q:g}: M_total = {t:.6g} M_earth, M_total/N = {t / n:.6g} M_earth")
    return fig

In [ ]:
USE_SLIDERS = False   # True: build the slopes from the section-7 sliders (all but q; M_disk slider = total mass)
SAVE_AS = "multi_slope"   # folder under mass_model_exports/; None = don't save
SLOPES = (2.5, 3.5, 4.5)

if USE_SLIDERS:
    params, disk, a_seed, md = slider_params(), current_disk(), D["a_seed"].value, m_disk()
else:   # the fixed configuration: MODEL_DEFAULTS, uniform a in [95, 105] AU, M_disk = 1 M_earth
    params = dict(MODEL_DEFAULTS)
    disk = dict(a_distribution="uniform", amin=95, amax=105, emax=3.2e-5, imax_deg=3.2e-5)
    a_seed, md = SIM_SEED, 1.0

# Steps 1-2: build once (~10 s: three 5M-body cascades), then rescale each slope to md
sets10 = build_slope_sets(SLOPES, params)
sets10_rescaled = {q: rescale_masses(k, md) for q, k in sets10.items()}
# Step 3: shared locations, run_simulation draw order
a_shared = semimajor_axes(len(next(iter(sets10.values()))), disk, seed=a_seed)
# Steps 4-5
plot_mass_vs_a_multi(sets10_rescaled, a_shared, total_mass_earth=md)
plt.show()
if SAVE_AS:
    save_slope_sets(sets10, SAVE_AS, a_shared, md, params, disk, a_seed)

## 11. Compare three models side by side

Three independent slider panels, **A**, **B** and **C**, each with its own full set of mass-model settings (slope, radius range, `n_keep`, selection,
density, cascade size, seed). Everything is **rescaled to the one shared M_disk** below, so the models are directly comparable; the raw (un-rescaled) totals are
in the table. The semimajor axes use the disk settings from section 7 (the first `n_keep` locations are shared by every model, so body *i* sits at the same `a` in each).

Starting values: q = 2.5 / 3.5 / 4.5 with seeds 42 / 43 / 44 (what `make_cascade_selection.py` uses), everything else from `MODEL_DEFAULTS`.
Below the summary plots, each model also gets the section-6 four-panel figure: cumulative size distribution (dropped / kept / below block vs analytic),
finite-sample departure, individual masses and where the mass is (masses rescaled to M_disk).

Press **Run comparison**. If "save selections" is ticked, each model is written to `mass_model_exports/<name>/model_<X>/selected.csv` (raw masses, `a_au`, `mass_earth_rescaled`)
plus `selection_summary.csv` and `params.json`.

In [ ]:
CMP_LABELS = ("A", "B", "C")
CMP_COLORS = ("tab:blue", "tab:orange", "tab:green")
CMP_Q = (2.5, 3.5, 4.5)


def model_panel(label, q, seed):
    """One column of sliders for a model; returns (dict of widgets, panel)."""
    m = MODEL_DEFAULTS
    ws = dict(
        q=fs(q, 1.5, 5.5, 0.1, "slope q"),
        radius_min=fs(m["radius_min"], 0.1, 50, 0.1, "R min (km)"),
        radius_max=fs(m["radius_max"], 50, 3000, 10, "R max (km)"),
        n_keep=isl(m["n_keep"], 50, 5000, 50, "n_keep"),
        selection=w.ToggleButtons(options=list(SELECTION_MODES), value=m["selection"], description="selection", style=style),
        n_drop=isl(m["n_drop"], 0, 1000, 10, "n_drop"),
        # tol: powerlaw selection only (ignored by drop_top); smaller = stricter = deeper cut. See find_cut_rank.
        tol=fs(m["tol"], 0.05, 0.5, 0.05, "tol (powerlaw)"),
        n_cascade=w.Dropdown(options=[100_000, 500_000, 1_000_000, 5_000_000, 10_000_000], value=m["n_cascade"], description="n_cascade", style=style),
        density_model=w.ToggleButtons(options=list(DENSITY_MODELS), value=m["density_model"], description="density", style=style),
        density=fs(m["density"], 0.5, 3.5, 0.1, "ρ const (g/cm³)"),
        seed=isl(seed, 0, 999, 1, "seed"),
    )
    color = CMP_COLORS[CMP_LABELS.index(label)]
    head = w.HTML(f"<b style='color:{color}'>Model {label}</b>")
    return ws, w.VBox([head, *ws.values()], layout=w.Layout(width="470px"))


CMP_W, panels = [], []
for lab, q, i in zip(CMP_LABELS, CMP_Q, range(3)):
    ws, panel = model_panel(lab, q, MODEL_DEFAULTS["seed"] + i)
    CMP_W.append(ws); panels.append(panel)

cmp_m_disk = fs(1.0, 0.01, 20, 0.01, "M_disk (all models)")
cmp_save = w.Checkbox(value=True, description="save selections", style=style)
cmp_name = w.Text(value="compare_three", description="save as", style=style)
cmp_btn = w.Button(description="Run comparison", button_style="primary", icon="play")
cmp_status = w.HTML()
cmp_table = w.HTML()
cmp_img = w.Image(format="png", layout=w.Layout(max_width="100%"))
# per-model detail figures (cumulative size distribution, finite-sample departure, individual masses, where the mass is)
cmp_model_heads = [w.HTML() for _ in CMP_LABELS]
cmp_model_imgs = [w.Image(format="png", layout=w.Layout(max_width="100%")) for _ in CMP_LABELS]
cmp_state = {}


def plot_three(models, a_full, md):
    """Four views of the three models, all with masses rescaled to ``md``."""
    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    ax = ax.ravel()
    ns = {lab: len(m["rescaled"]) for lab, m in models.items()}
    for (lab, m), col in zip(models.items(), CMP_COLORS):
        df, name = m["rescaled"], f"{lab}: q={m['params']['q']:g}"
        rank = np.arange(1, len(df) + 1)
        ax[0].plot(df.sort_values("radius_km", ascending=False).radius_km, rank, c=col, label=name)
        o = df.sort_values("mass_earth", ascending=False)
        ax[1].plot(o.mass_earth, rank, c=col, label=name)
        ax[2].scatter(a_full[:len(df)], df.mass_earth, s=10, alpha=.6, color=col, label=name)
        ax[3].plot(rank, np.cumsum(o.mass_earth.values) / o.mass_earth.sum(), c=col, label=name)

    ax[0].set(xscale="log", yscale="log", xlabel="Radius (km)", ylabel="N(≥R)", title="Cumulative size distribution")
    add_radius_references(ax[0], axis="x", pad_factor=1.0, fontsize=6)
    ax[1].set(xscale="log", yscale="log", xlabel=f"Mass (Earth masses), rescaled to M_disk = {md:g}", ylabel="N(≥m)",
              title="Cumulative mass distribution")
    add_mass_references(ax[1], axis="x")
    # uniform control: M_disk / N (one red line if every model has the same N, else one dashed line per model)
    if len(set(ns.values())) == 1:
        n = next(iter(ns.values()))
        ax[2].scatter(a_full[:n], np.full(n, md / n), s=6, marker="s", color="tab:red", alpha=.7,
                      label=f"uniform: M_disk/N = {md / n:.3g} ({n} bodies)")
    else:
        for (lab, m), col in zip(models.items(), CMP_COLORS):
            ax[2].axhline(md / ns[lab], ls="--", lw=1.2, color=col)
        ax[2].plot([], [], ls="--", c="0.3", label="M_disk/N (dashed, per model)")
    ax[2].set(yscale="log", xlabel="Semimajor axis (AU)", ylabel="Mass (Earth masses)", title="Mass vs semimajor axis (rescaled)")
    add_mass_references(ax[2])
    ax[3].set(xscale="log", xlabel="k  (k most massive bodies)", ylabel="fraction of M_disk",
              title="Mass concentration", ylim=(0, 1.02))
    for x in ax:
        x.grid(alpha=.3, which="both"); x.legend(fontsize=8)
    fig.tight_layout()
    return fig


def models_table(models, md):
    rows = []
    for lab, m in models.items():
        raw, r, p = m["raw"], m["rescaled"], m["params"]
        rows.append({
            "model": lab, "q": f"{p['q']:g}", "selection": p["selection"], "N": len(raw),
            "R range (km)": f"{raw.radius_km.min():.4g} – {raw.radius_km.max():.4g}",
            "mass range (M_earth, rescaled)": f"{r.mass_earth.min():.3g} – {r.mass_earth.max():.3g}",
            "largest body / M_disk": f"{r.mass_earth.max() / md:.2%}",
            "M_disk/N": f"{md / len(r):.3g}",
            "raw total (M_earth)": f"{raw.mass_earth.sum():.4g}",
            "rescale factor": f"{r.attrs['mass_scale']:.4g}",
            "ranks kept": f"{raw.attrs['rank_window'][0]}–{raw.attrs['rank_window'][1]}",
        })
    return pd.DataFrame(rows).to_html(index=False, border=0)


def save_models(models, name, a_full, md, disk, a_seed, outdir="mass_model_exports"):
    """Write each model as model_<X>/selected.csv (+ a_au, mass_earth_rescaled), selection_summary.csv and params.json."""
    root = Path(outdir) / name
    rows = []
    for lab, m in models.items():
        d = root / f"model_{lab}"
        d.mkdir(parents=True, exist_ok=True)
        df = m["raw"].copy()
        df["a_au"] = a_full[:len(df)]
        df["mass_earth_rescaled"] = m["rescaled"]["mass_earth"].values
        df.to_csv(d / "selected.csv", index=False)
        rows.append(dict(model=lab, slope=m["params"]["q"], cut_rank=m["raw"].attrs["rank_window"][0],
                         keep_r_min=m["raw"].radius_km.min(), keep_r_max=m["raw"].radius_km.max(),
                         raw_total_mass_earth=m["raw"].mass_earth.sum(), m_disk_rescaled=md))
    pd.DataFrame(rows).to_csv(root / "selection_summary.csv", index=False)
    with open(root / "params.json", "w") as f:
        json.dump(dict(models={lab: m["params"] for lab, m in models.items()}, m_disk_rescaled=md, disk=disk, a_seed=a_seed),
                  f, indent=2, default=str)
    return root


def run_compare(_=None):
    global models3
    md = cmp_m_disk.value
    cmp_status.value = "<i>running… (each model draws its own cascade)</i>"
    try:
        models = {}
        for lab, ws in zip(CMP_LABELS, CMP_W):
            p = {k: v.value for k, v in ws.items()}
            raw, cfg_i, rd_i, ks_i, ratio_i = build_mass_model(**p)
            models[lab] = dict(params=p, raw=raw, rescaled=rescale_masses(raw, md), cfg=cfg_i,
                               radii_desc=rd_i, k_start=ks_i, ratio=ratio_i)
        disk, a_seed = current_disk(), D["a_seed"].value
        a_full = semimajor_axes(max(len(m["raw"]) for m in models.values()), disk, seed=a_seed)
        models3 = models
        cmp_state.update(models=models, a=a_full, m_disk=md, disk=disk)
        cmp_img.value = fig_png(plot_three(models, a_full, md))
        # per-model detail: the section-6 four-panel figure (masses rescaled to M_disk; radii/ranks unchanged)
        for img, head, (lab, m) in zip(cmp_model_imgs, cmp_model_heads, models.items()):
            q = m["params"]["q"]
            head.value = f"<h4 style='color:{CMP_COLORS[CMP_LABELS.index(lab)]}'>Model {lab}: q = {q:g}, {m['params']['selection']}</h4>"
            img.value = fig_png(plot_model(m["rescaled"], m["cfg"], m["radii_desc"], m["k_start"], m["ratio"], q,
                                           title=f"Model {lab}: q = {q:g}  (masses rescaled to M_disk = {md:g} M_earth)"))
        cmp_table.value = models_table(models, md)
        msg = f"<pre>{SemimajorAxisSampler(disk).describe()}</pre>"
        if cmp_save.value and cmp_name.value.strip():
            root = save_models(models, cmp_name.value.strip(), a_full, md, disk, a_seed)
            msg += f"<pre>Saved selections to: {root}</pre>"
        cmp_status.value = msg
    except Exception as e:
        cmp_status.value = f"<pre style='color:crimson'>Error: {e}</pre>"

cmp_btn.on_click(run_compare)
display(w.VBox([
    w.HBox(panels, layout=w.Layout(flex_flow="row wrap")),
    cmp_m_disk, w.HBox([cmp_save, cmp_name]), cmp_btn, cmp_status, cmp_table, cmp_img,
    *[x for pair in zip(cmp_model_heads, cmp_model_imgs) for x in pair],
]))
run_compare()